<b> This code belongs to the author Nayan Talmale </b>

<b> Import the libraries </b>

In [32]:
import warnings 
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
import glob
import xarray
import multiprocessing as mp 
from dask.diagnostics import ProgressBar

import xesmf as xe

<b> Step 1: First find all the models having respective variables </b>

In [7]:
# Function to find all files and create dict
def find_all_files(pattern):
    all_paths = sorted(glob.glob(pattern))
    model_files = {}
    for path in all_paths:
        path_parts = path.split('/')
        model_identifier = path_parts[8]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'
    return model_files

In [8]:
# Patterns
tas_pattern_hist    = f'/g/data/**/**/CMIP6/CMIP/*/*/historical/r1i*/Amon/tas/gn/*/*.nc'
tasmax_pattern_hist = f'/g/data/**/**/CMIP6/CMIP/*/*/historical/r1i*/Amon/tasmax/gn/*/*.nc'
tasmin_pattern_hist = f'/g/data/**/**/CMIP6/CMIP/*/*/historical/r1i*/Amon/tasmin/gn/*/*.nc'
pr_pattern_hist     = f'/g/data/**/**/CMIP6/CMIP/*/*/historical/r1i*/Amon/pr/gn/*/*.nc'

# Find files per variable
tas_files    = find_all_files(tas_pattern_hist)
tasmax_files = find_all_files(tasmax_pattern_hist)
tasmin_files = find_all_files(tasmin_pattern_hist)
pr_files     = find_all_files(pr_pattern_hist)

print(f"tas:    {len(tas_files)} models")
print(f"tasmax: {len(tasmax_files)} models")
print(f"tasmin: {len(tasmin_files)} models")
print(f"pr:     {len(pr_files)} models")

tas:    44 models
tasmax: 29 models
tasmin: 29 models
pr:     42 models


In [9]:
tas_files

{'ACCESS-CM2': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/Amon/tas/gn/latest/*.nc',
 'ACCESS-ESM1-5': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r1i1p1f1/Amon/tas/gn/latest/*.nc',
 'TaiESM1': '/g/data/oi10/replicas/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/tas/gn/v20200218/*.nc',
 'AWI-CM-1-1-MR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-CM-1-1-MR/historical/r1i1p1f1/Amon/tas/gn/v20191015/*.nc',
 'AWI-ESM-1-1-LR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/Amon/tas/gn/v20200212/*.nc',
 'BCC-CSM2-MR': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-CSM2-MR/historical/r1i1p1f1/Amon/tas/gn/v20181126/*.nc',
 'BCC-ESM1': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-ESM1/historical/r1i1p1f1/Amon/tas/gn/v20181214/*.nc',
 'CAMS-CSM1-0': '/g/data/oi10/replicas/CMIP6/CMIP/CAMS/CAMS-CSM1-0/historical/r1i1p1f1/Amon/tas/gn/v20190708/*.nc',
 'CAS-ESM2-0': '/g/data/oi10/replicas/CMIP6/CMIP/CAS/CAS-ESM2-0/

In [10]:
tasmax_files

{'ACCESS-CM2': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/Amon/tasmax/gn/latest/*.nc',
 'ACCESS-ESM1-5': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r1i1p1f1/Amon/tasmax/gn/latest/*.nc',
 'AWI-CM-1-1-MR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-CM-1-1-MR/historical/r1i1p1f1/Amon/tasmax/gn/v20181218/*.nc',
 'AWI-ESM-1-1-LR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/Amon/tasmax/gn/v20200212/*.nc',
 'BCC-CSM2-MR': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-CSM2-MR/historical/r1i1p1f1/Amon/tasmax/gn/v20181126/*.nc',
 'BCC-ESM1': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-ESM1/historical/r1i1p1f1/Amon/tasmax/gn/v20181214/*.nc',
 'CAS-ESM2-0': '/g/data/oi10/replicas/CMIP6/CMIP/CAS/CAS-ESM2-0/historical/r1i1p1f1/Amon/tasmax/gn/v20201227/*.nc',
 'FGOALS-g3': '/g/data/oi10/replicas/CMIP6/CMIP/CAS/FGOALS-g3/historical/r1i1p1f1/Amon/tasmax/gn/v20190818/*.nc',
 'CanESM5-1': '/g/data/oi10/replicas/CMIP6/

In [11]:
tasmin_files

{'ACCESS-CM2': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/Amon/tasmin/gn/latest/*.nc',
 'ACCESS-ESM1-5': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r1i1p1f1/Amon/tasmin/gn/latest/*.nc',
 'AWI-CM-1-1-MR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-CM-1-1-MR/historical/r1i1p1f1/Amon/tasmin/gn/v20181218/*.nc',
 'AWI-ESM-1-1-LR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/Amon/tasmin/gn/v20200212/*.nc',
 'BCC-CSM2-MR': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-CSM2-MR/historical/r1i1p1f1/Amon/tasmin/gn/v20181126/*.nc',
 'BCC-ESM1': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-ESM1/historical/r1i1p1f1/Amon/tasmin/gn/v20181214/*.nc',
 'CAS-ESM2-0': '/g/data/oi10/replicas/CMIP6/CMIP/CAS/CAS-ESM2-0/historical/r1i1p1f1/Amon/tasmin/gn/v20201227/*.nc',
 'FGOALS-g3': '/g/data/oi10/replicas/CMIP6/CMIP/CAS/FGOALS-g3/historical/r1i1p1f1/Amon/tasmin/gn/v20190818/*.nc',
 'CanESM5-1': '/g/data/oi10/replicas/CMIP6/

In [12]:
pr_files

{'ACCESS-CM2': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/Amon/pr/gn/latest/*.nc',
 'ACCESS-ESM1-5': '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r1i1p1f1/Amon/pr/gn/latest/*.nc',
 'TaiESM1': '/g/data/oi10/replicas/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/pr/gn/v20200218/*.nc',
 'AWI-CM-1-1-MR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-CM-1-1-MR/historical/r1i1p1f1/Amon/pr/gn/v20200511/*.nc',
 'AWI-ESM-1-1-LR': '/g/data/oi10/replicas/CMIP6/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/Amon/pr/gn/v20200212/*.nc',
 'BCC-CSM2-MR': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-CSM2-MR/historical/r1i1p1f1/Amon/pr/gn/v20181126/*.nc',
 'BCC-ESM1': '/g/data/oi10/replicas/CMIP6/CMIP/BCC/BCC-ESM1/historical/r1i1p1f1/Amon/pr/gn/v20181214/*.nc',
 'CAMS-CSM1-0': '/g/data/oi10/replicas/CMIP6/CMIP/CAMS/CAMS-CSM1-0/historical/r1i1p1f1/Amon/pr/gn/v20190708/*.nc',
 'CAS-ESM2-0': '/g/data/oi10/replicas/CMIP6/CMIP/CAS/CAS-ESM2-0/historic

<b> Step 2- Load the historical data from 1850-2014 </b>

In [17]:
def process_model_var(model_identifier, files_dict, variable):
    try:
        var_file_hist = files_dict[model_identifier]

        ds_var_hist = xr.open_mfdataset(var_file_hist, use_cftime=True)

        # Drop bounds variables that can cause size-mismatch errors after slicing
        bnds_vars = [v for v in ['time_bnds', 'lat_bnds', 'lon_bnds'] if v in ds_var_hist.variables]
        ds_var_hist = ds_var_hist.drop_vars(bnds_vars)

        ds_var_hist = ds_var_hist.sel(time=slice('1850-01-01', '2015-01-01'))
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')

        regridder = xe.Regridder(ds_var_hist, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)
        var = regridder(ds_var_hist[variable].load())

        return model_identifier, var
    except Exception as e:
        print(f"Error processing {model_identifier} ({variable}): {e}")
        return None  # explicit, so we can filter these out later


def build_dataset(process_fn, variable_name):
    res_arr = []
    with mp.Pool(processes=8) as pool:
        i = 0
        for res in pool.imap(process_fn, models_to_process):
            res_arr.append(res)
            print(f'{variable_name}: Completed {i+1}/{len(models_to_process)}', end='\r')
            i += 1
    print()

    # Filter out failed models (None results)
    successful = [r for r in res_arr if r is not None]
    failed_models = [models_to_process[i] for i, r in enumerate(res_arr) if r is None]
    if failed_models:
        print(f"{variable_name}: dropped models due to errors: {failed_models}")

    model_names_ok = [r[0] for r in successful]
    data_arrays_ok = [r[1] for r in successful]

    ds = xr.concat(
        data_arrays_ok,
        dim=model_names_ok,
        coords='minimal',
        compat='override'
    ).rename(dict(concat_dim='model_en')).to_dataset(name=variable_name)

    return ds

In [18]:
# ============================================================
# Build all four datasets, using the SAME model list
# ============================================================
CMIP6_tasmax_hist = build_dataset(process_tasmax, 'tasmax')
CMIP6_tasmin_hist = build_dataset(process_tasmin, 'tasmin')
CMIP6_tas_hist    = build_dataset(process_tas, 'tas')
CMIP6_pr_hist     = build_dataset(process_pr, 'pr')

print(CMIP6_tasmax_hist)
print(CMIP6_tasmin_hist)
print(CMIP6_tas_hist)
print(CMIP6_pr_hist)

tasmax: Completed 28/28
tasmin: Completed 28/28
Error processing AWI-CM-1-1-MR (tas): conflicting sizes for dimension 'time': length 1980 on 'time' and length 1968 on {'time': 'tas', 'lat': 'tas', 'lon': 'tas'}
tas: Completed 28/28
tas: dropped models due to errors: ['AWI-CM-1-1-MR']
pr: Completed 28/28
<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
    height    float64 8B 2.0
Data variables:
    tasmax    (model_en, time, lat, lon) float32 6GB 253.8 253.7 ... 240.5 240.5
<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-3

In [19]:
CMIP6_tasmax_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
    height    float64 8B 2.0
Data variables:
    tasmax    (model_en, time, lat, lon) float32 6GB 253.8 253.7 ... 240.5 240.5

In [20]:
CMIP6_tasmin_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
    height    float64 8B 2.0
Data variables:
    tasmin    (model_en, time, lat, lon) float32 6GB 251.7 251.6 ... 236.2 236.2

In [21]:
CMIP6_tas_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 27)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 216B 'ACCESS-CM2' ... 'SAM0-UNICON'
    height    float64 8B 2.0
Data variables:
    tas       (model_en, time, lat, lon) float32 6GB 252.8 252.7 ... 238.2 238.3

In [22]:
CMIP6_pr_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
Data variables:
    pr        (model_en, time, lat, lon) float32 6GB 2.581e-06 ... 3e-06

<b> Now lets see/get SSPs data as well </b>

In [23]:
scenarios = ['ssp126', 'ssp245', 'ssp370', 'ssp585']
variables = ['tas', 'tasmax', 'tasmin', 'pr']

ssp_files = {}  # ssp_files[scenario][variable] = {model: pattern}

for scenario in scenarios:
    ssp_files[scenario] = {}
    for var in variables:
        pattern = f'/g/data/**/**/CMIP6/ScenarioMIP/*/*/{scenario}/r1i*/Amon/{var}/gn/*/*.nc'
        files = find_all_files(pattern)
        ssp_files[scenario][var] = files
        print(f"{scenario} - {var}: {len(files)} models")
    print()

ssp126 - tas: 29 models
ssp126 - tasmax: 19 models
ssp126 - tasmin: 19 models
ssp126 - pr: 28 models

ssp245 - tas: 28 models
ssp245 - tasmax: 20 models
ssp245 - tasmin: 20 models
ssp245 - pr: 28 models

ssp370 - tas: 26 models
ssp370 - tasmax: 19 models
ssp370 - tasmin: 19 models
ssp370 - pr: 27 models

ssp585 - tas: 31 models
ssp585 - tasmax: 24 models
ssp585 - tasmin: 24 models
ssp585 - pr: 31 models



In [24]:
scenario = 'ssp534-over'
variables = ['tas', 'tasmax', 'tasmin', 'pr']

os_files = {}
for var in variables:
    pattern = f'/g/data/**/**/CMIP6/ScenarioMIP/*/*/{scenario}/r1i*/Amon/{var}/gn/*/*.nc'
    files = find_all_files(pattern)
    os_files[var] = files
    print(f"{scenario} - {var}: {len(files)} models")
    for m in files:
        print(f"   {m}")

ssp534-over - tas: 15 models
   ACCESS-CM2
   ACCESS-ESM1-5
   AWI-ESM-1-REcoM
   FGOALS-g3
   CanESM5
   CMCC-ESM2
   MIROC-ES2L
   MIROC6
   UKESM1-0-LL
   MRI-ESM2-0
   GISS-E2-1-G
   GISS-E2-1-H
   GISS-E2-2-G
   CESM2-WACCM
   NorESM2-LM
ssp534-over - tasmax: 2 models
   ACCESS-CM2
   ACCESS-ESM1-5
ssp534-over - tasmin: 2 models
   ACCESS-CM2
   ACCESS-ESM1-5
ssp534-over - pr: 14 models
   ACCESS-CM2
   ACCESS-ESM1-5
   AWI-ESM-1-REcoM
   FGOALS-g3
   CanESM5
   CMCC-ESM2
   MIROC-ES2L
   MIROC6
   UKESM1-0-LL
   MRI-ESM2-0
   GISS-E2-1-G
   GISS-E2-1-H
   GISS-E2-2-G
   CESM2-WACCM


In [25]:
scenario = 'ssp119'
variables = ['tas', 'tasmax', 'tasmin', 'pr']

os_files_ssp119 = {}
for var in variables:
    pattern = f'/g/data/**/**/CMIP6/ScenarioMIP/*/*/{scenario}/r1i*/Amon/{var}/gn/*/*.nc'
    files = find_all_files(pattern)
    os_files_ssp119[var] = files
    print(f"{scenario} - {var}: {len(files)} models")
    for m in files:
        print(f"   {m}")

ssp119 - tas: 8 models
   CAMS-CSM1-0
   FGOALS-g3
   CanESM5
   MIROC-ES2L
   MIROC6
   UKESM1-0-LL
   MRI-ESM2-0
   GISS-E2-1-G
ssp119 - tasmax: 0 models
ssp119 - tasmin: 0 models
ssp119 - pr: 8 models
   CAMS-CSM1-0
   FGOALS-g3
   CanESM5
   MIROC-ES2L
   MIROC6
   UKESM1-0-LL
   MRI-ESM2-0
   GISS-E2-1-G


<b> Summary </b>

In [27]:
scenarios = ['ssp119', 'ssp126', 'ssp245', 'ssp370', 'ssp585', 'ssp534-over']
variables = ['tas', 'tasmax', 'tasmin', 'pr']

ssp_files = {}  # ssp_files[scenario][variable] = {model: pattern}
counts = []      # for building the summary dataframe

for scenario in scenarios:
    ssp_files[scenario] = {}
    for var in variables:
        pattern = f'/g/data/**/**/CMIP6/ScenarioMIP/*/*/{scenario}/r1i*/Amon/{var}/gn/*/*.nc'
        files = find_all_files(pattern)
        ssp_files[scenario][var] = files
        counts.append({'scenario': scenario, 'variable': var, 'n_models': len(files)})

# Build a clean summary table: scenarios as rows, variables as columns
df_counts = pd.DataFrame(counts)
summary_table = df_counts.pivot(index='scenario', columns='variable', values='n_models')
summary_table = summary_table[['tas', 'tasmax', 'tasmin', 'pr']]  # fixed column order
summary_table = summary_table.reindex(scenarios)  # fixed row order

print(summary_table)

variable     tas  tasmax  tasmin  pr
scenario                            
ssp119         8       0       0   8
ssp126        29      19      19  28
ssp245        28      20      20  28
ssp370        26      19      19  27
ssp585        31      24      24  31
ssp534-over   15       2       2  14


<b> Now work on Historical </b>

In [28]:
CMIP6_tasmax_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
    height    float64 8B 2.0
Data variables:
    tasmax    (model_en, time, lat, lon) float32 6GB 253.8 253.7 ... 240.5 240.5

In [29]:
CMIP6_tasmin_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
    height    float64 8B 2.0
Data variables:
    tasmin    (model_en, time, lat, lon) float32 6GB 251.7 251.6 ... 236.2 236.2

In [30]:
CMIP6_pr_hist

<xarray.Dataset> Size: 6GB
Dimensions:   (time: 1980, lon: 240, lat: 120, model_en: 28)
Coordinates:
  * time      (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
  * lon       (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
  * lat       (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model_en  (model_en) object 224B 'ACCESS-CM2' ... 'SAM0-UNICON'
Data variables:
    pr        (model_en, time, lat, lon) float32 6GB 2.581e-06 ... 3e-06

<b> Preparing our Objective data </b>

<b> Step 1: - Unit Conversion </b>

In [33]:
# ============================================================
# Temperature: Kelvin -> Celsius
# ============================================================
CMIP6_tasmax_hist['tasmax'] = CMIP6_tasmax_hist['tasmax'] - 273.15
CMIP6_tasmax_hist['tasmax'].attrs['units'] = 'degC'

CMIP6_tasmin_hist['tasmin'] = CMIP6_tasmin_hist['tasmin'] - 273.15
CMIP6_tasmin_hist['tasmin'].attrs['units'] = 'degC'

CMIP6_tas_hist['tas'] = CMIP6_tas_hist['tas'] - 273.15
CMIP6_tas_hist['tas'].attrs['units'] = 'degC'


# ============================================================
# Precipitation: kg m-2 s-1 -> mm/month
# ============================================================
# Get number of days in each month, from the actual time coordinate
days_in_month = CMIP6_pr_hist['time'].dt.days_in_month
seconds_in_month = days_in_month * 86400  # 86400 seconds per day

CMIP6_pr_hist['pr'] = CMIP6_pr_hist['pr'] * seconds_in_month
CMIP6_pr_hist['pr'].attrs['units'] = 'mm/month'

<b> Step 2: - Make this data seasonal and get 15 year rolling mean </b>

<b> 1. Seasonal </b>

In [36]:
seasons = {
    'DJF': [12, 1, 2],
    'JJA': [6, 7, 8],
}

def select_season(da, month):
    """
    Select months belonging to a season and resample to one value per season-year.
    """
    da_season = da.sel(time=da.time.dt.month.isin(month)).resample(time='AS-JUN').mean('time')
    return da_season

In [38]:
tasmax_djf = select_season(CMIP6_tasmax_hist['tasmax'], month=seasons['DJF'])
tasmax_jja = select_season(CMIP6_tasmax_hist['tasmax'], month=seasons['JJA'])

tasmin_djf = select_season(CMIP6_tasmin_hist['tasmin'], month=seasons['DJF'])
tasmin_jja = select_season(CMIP6_tasmin_hist['tasmin'], month=seasons['JJA'])

pr_djf = select_season(CMIP6_pr_hist['pr'], month=seasons['DJF'])
pr_jja = select_season(CMIP6_pr_hist['pr'], month=seasons['JJA'])

tas_djf = select_season(CMIP6_tas_hist['tas'], month=seasons['DJF'])
tas_jja = select_season(CMIP6_tas_hist['tas'], month=seasons['JJA'])

print(tasmax_djf)
print(tasmin_djf)
print(pr_djf)
print(tas_djf)

<xarray.DataArray 'tasmax' (model_en: 28, time: 166, lat: 120, lon: 240)> Size: 535MB
array([[[[-28.521957 , -28.643234 , -28.635345 , ..., -28.47493  ,
          -28.495209 , -28.426636 ],
         [-29.306847 , -29.356903 , -29.386147 , ..., -29.084274 ,
          -29.147568 , -29.23352  ],
         [-27.509186 , -27.621078 , -27.705933 , ..., -27.181053 ,
          -27.295792 , -27.393333 ],
         ...,
         [-33.37079  , -33.38231  , -33.388596 , ..., -33.24578  ,
          -33.314026 , -33.35016  ],
         [-34.048027 , -34.088745 , -34.147728 , ..., -33.791504 ,
          -33.906036 , -33.998344 ],
         [-34.174744 , -34.184433 , -34.272034 , ..., -34.153282 ,
          -34.196854 , -34.20066  ]],

        [[-23.485657 , -23.579123 , -23.593424 , ..., -23.41976  ,
          -23.422562 , -23.405325 ],
         [-23.487823 , -23.480783 , -23.501688 , ..., -23.363052 ,
          -23.409052 , -23.476013 ],
         [-21.934484 , -22.058142 , -22.164637 , ..., -21.543081 ,

<b> 2. Rolling mean </b>

In [39]:
def trim_and_detrend(da_season, start='1940-01-16', end='2014-12-31', window=15):
    """
    Trim an already season-resampled DataArray to the analysis window,
    then compute the anomaly relative to a centered 15-year rolling mean.
    """
    with ProgressBar():
        da_season = da_season.sel(time=slice(start, end)).compute()

    da_detrend = da_season - da_season.rolling(time=window, center=True).mean('time')

    return da_detrend

In [40]:
tasmax_djf_detrend = trim_and_detrend(tasmax_djf)
tasmax_jja_detrend = trim_and_detrend(tasmax_jja)

tasmin_djf_detrend = trim_and_detrend(tasmin_djf)
tasmin_jja_detrend = trim_and_detrend(tasmin_jja)

pr_djf_detrend = trim_and_detrend(pr_djf)
pr_jja_detrend = trim_and_detrend(pr_jja)

tas_djf_detrend = trim_and_detrend(tas_djf)
tas_jja_detrend = trim_and_detrend(tas_jja)

<b> Lets drop the nan values as rolling mean will miss </b>

In [46]:
# Drop leading/trailing NaN years (from centered 15-year rolling window edges)
tasmax_djf_detrend = tasmax_djf_detrend.dropna(dim='time', how='all')
tasmax_jja_detrend = tasmax_jja_detrend.dropna(dim='time', how='all')

tasmin_djf_detrend = tasmin_djf_detrend.dropna(dim='time', how='all')
tasmin_jja_detrend = tasmin_jja_detrend.dropna(dim='time', how='all')

pr_djf_detrend = pr_djf_detrend.dropna(dim='time', how='all')
pr_jja_detrend = pr_jja_detrend.dropna(dim='time', how='all')

tas_djf_detrend = tas_djf_detrend.dropna(dim='time', how='all')
tas_jja_detrend = tas_jja_detrend.dropna(dim='time', how='all')

# Quick check
print(f"tasmax_djf_detrend: {len(tasmax_djf_detrend.time)} time steps")
print(f"tasmax_jja_detrend: {len(tasmax_jja_detrend.time)} time steps")
print(f"tasmin_djf_detrend: {len(tasmin_djf_detrend.time)} time steps")
print(f"tasmin_jja_detrend: {len(tasmin_jja_detrend.time)} time steps")
print(f"pr_djf_detrend:     {len(pr_djf_detrend.time)} time steps")
print(f"pr_jja_detrend:     {len(pr_jja_detrend.time)} time steps")
print(f"tas_djf_detrend:    {len(tas_djf_detrend.time)} time steps")
print(f"tas_jja_detrend:    {len(tas_jja_detrend.time)} time steps")

tasmax_djf_detrend: 61 time steps
tasmax_jja_detrend: 61 time steps
tasmin_djf_detrend: 61 time steps
tasmin_jja_detrend: 61 time steps
pr_djf_detrend:     61 time steps
pr_jja_detrend:     61 time steps
tas_djf_detrend:    61 time steps
tas_jja_detrend:    61 time steps


<b> Save these data </b>

In [47]:
output_dir = '/g/data/gb02/nt4273/Data/CMIP6_Hist_tasmax_tasmin_tas_pr/'

tasmax_djf_detrend.to_dataset(name='tasmax').to_netcdf(f'{output_dir}/tasmax_djf_detrend.nc')
tasmax_jja_detrend.to_dataset(name='tasmax').to_netcdf(f'{output_dir}/tasmax_jja_detrend.nc')

tasmin_djf_detrend.to_dataset(name='tasmin').to_netcdf(f'{output_dir}/tasmin_djf_detrend.nc')
tasmin_jja_detrend.to_dataset(name='tasmin').to_netcdf(f'{output_dir}/tasmin_jja_detrend.nc')

pr_djf_detrend.to_dataset(name='pr').to_netcdf(f'{output_dir}/pr_djf_detrend.nc')
pr_jja_detrend.to_dataset(name='pr').to_netcdf(f'{output_dir}/pr_jja_detrend.nc')

tas_djf_detrend.to_dataset(name='tas').to_netcdf(f'{output_dir}/tas_djf_detrend.nc')
tas_jja_detrend.to_dataset(name='tas').to_netcdf(f'{output_dir}/tas_jja_detrend.nc')

print("All files saved to", output_dir)
print(os.listdir(output_dir))

All files saved to /g/data/gb02/nt4273/Data/CMIP6_Hist_tasmax_tasmin_tas_pr/
['tas_jja_detrend.nc', 'tas_djf_detrend.nc', 'tasmin_djf_detrend.nc', 'pr_jja_detrend.nc', 'tasmax_jja_detrend.nc', 'tasmax_djf_detrend.nc', 'pr_djf_detrend.nc', 'tasmin_jja_detrend.nc']


<b> Now also create IPCC AR6 region data </b>

In [52]:
# Let's get the IPCC AR6 regions and points only over land

import regionmask

lon = tasmax_djf_detrend['lon']
lat = tasmax_djf_detrend['lat']

ar6_land = regionmask.defined_regions.ar6.land
keep = [i for i in range(len(ar6_land.numbers)) if i not in [44, 45]]
ar6_land_no_ant = regionmask.Regions(
    outlines=[ar6_land.polygons[i] for i in keep],
    numbers=[ar6_land.numbers[i] for i in keep],
    names=[ar6_land.names[i] for i in keep],
    abbrevs=[ar6_land.abbrevs[i] for i in keep],
    name="AR6 Land (No Antarctica)"
)

land_50 = regionmask.defined_regions.natural_earth_v5_0_0.land_50
land_mask_3D = land_50.mask_3D(lon, lat)
land_mask = land_mask_3D.isel(region=0)

mask_3D = ar6_land_no_ant.mask_3D(lon, lat)

mask_3D_land = mask_3D & land_mask

weights = np.cos(np.deg2rad(lat))

print(mask_3D_land)

<xarray.DataArray 'mask' (region: 44, lat: 120, lon: 240)> Size: 1MB
array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
...
        [False, False, False,

In [53]:
def to_regional(da):
    """Collapse (model_en, time, lat, lon) -> (model_en, time, region) via area-weighted land mask."""
    return da.where(mask_3D_land).weighted(weights).mean(dim=['lat', 'lon'])

tasmax_djf_regional = to_regional(tasmax_djf_detrend)
tasmax_jja_regional = to_regional(tasmax_jja_detrend)

tasmin_djf_regional = to_regional(tasmin_djf_detrend)
tasmin_jja_regional = to_regional(tasmin_jja_detrend)

pr_djf_regional = to_regional(pr_djf_detrend)
pr_jja_regional = to_regional(pr_jja_detrend)

tas_djf_regional = to_regional(tas_djf_detrend)
tas_jja_regional = to_regional(tas_jja_detrend)

In [54]:
print(tasmax_djf_regional)

<xarray.DataArray 'tasmax' (model_en: 28, time: 61, region: 44)> Size: 601kB
array([[[ 0.46756536, -1.12721995, -2.52107122, ...,  0.39017272,
          0.44272383, -0.62315546],
        [-3.08080965, -0.3285647 , -0.40984321, ..., -1.19732898,
         -1.10410635, -0.34694058],
        [ 2.45600763, -2.94421398,  0.43673732, ...,  0.21649753,
          0.1274235 ,  0.20682106],
        ...,
        [-1.06630562,  0.85552519,  0.92485684, ...,  3.99344458,
          0.99289426, -1.86119952],
        [-0.71262502, -0.65566883, -0.18507682, ..., -3.78377708,
         -2.43274936,  0.06478816],
        [-1.35514988,  0.58058595,  0.01407243, ...,  0.47095291,
          0.61919619,  0.1867276 ]],

       [[ 2.33627384, -2.53579208,  0.69680228, ...,  0.48158285,
          0.01733561,  0.29141622],
        [ 0.32701524,  1.52082745,  1.89023038, ...,  0.07162255,
          0.54689428, -0.15206162],
        [ 0.47153759, -0.0448855 , -1.3283342 , ..., -0.75371354,
         -1.17097587,  0.1

In [55]:
output_dir = '/g/data/gb02/nt4273/Data/CMIP6_Hist_tasmax_tasmin_tas_pr/'

tasmax_djf_regional.to_dataset(name='tasmax').to_netcdf(f'{output_dir}/tasmax_djf_regional.nc')
tasmax_jja_regional.to_dataset(name='tasmax').to_netcdf(f'{output_dir}/tasmax_jja_regional.nc')

tasmin_djf_regional.to_dataset(name='tasmin').to_netcdf(f'{output_dir}/tasmin_djf_regional.nc')
tasmin_jja_regional.to_dataset(name='tasmin').to_netcdf(f'{output_dir}/tasmin_jja_regional.nc')

pr_djf_regional.to_dataset(name='pr').to_netcdf(f'{output_dir}/pr_djf_regional.nc')
pr_jja_regional.to_dataset(name='pr').to_netcdf(f'{output_dir}/pr_jja_regional.nc')

tas_djf_regional.to_dataset(name='tas').to_netcdf(f'{output_dir}/tas_djf_regional.nc')
tas_jja_regional.to_dataset(name='tas').to_netcdf(f'{output_dir}/tas_jja_regional.nc')

print("Saved:", os.listdir(output_dir))

Saved: ['tas_jja_detrend.nc', 'tas_djf_detrend.nc', 'tasmin_djf_detrend.nc', 'pr_jja_detrend.nc', 'tasmax_djf_regional.nc', 'tas_djf_regional.nc', 'tasmax_jja_detrend.nc', 'pr_djf_regional.nc', 'tasmax_djf_detrend.nc', 'pr_djf_detrend.nc', 'pr_jja_regional.nc', 'tas_jja_regional.nc', 'tasmin_jja_detrend.nc', 'tasmin_jja_regional.nc', 'tasmax_jja_regional.nc', 'tasmin_djf_regional.nc']
